# **Практическая работа №4. Работа с векторными данными в GeoPandas**

Выполняя практическую работу, опирайтесь на теоритический материал, рассматриваемый на занятии:

- Использование NumPy, Pandas и GeoPandas для работы с пространственными данными: https://u.to/rk86Ig
- Ультимативный обзор аналитических инструментов GeoPandas:  https://u.to/s0d3Ig
- Теоретические аспекты и практические примеры получения векторных данных из OpenStreetMap: https://u.to/9SdwIQ

Дополнительно:

- Особенности построения интерактивных карт с помощью библиотеки Leafmap: https://u.to/WvA3Ig

Официальная техническая документацию по используемым библиотекам:

- **NumPy**: https://numpy.org/doc/stable/
- **Pandas**: https://pandas.pydata.org/docs/
- **GeoPandas**: https://geopandas.org/en/stable/docs.html
- **Leafmap**: https://leafmap.org/

In [ ]:
%%capture
!pip install geopandas leafmap mapclassify # Устанавливаем библиотеку GeoPandas и необходимые зависимости

## **Задание №1. Операции с массивами NumPy и геопространственными координатами**


1. Создайте двумерный массив NumPy, содержащий широту и долготу следующих городов: Токио (35.6895, 139.6917), Нью-Йорк (40.7128, -74.0060), Лондон (51.5074, -0.1278) и Париж (48.8566, 2.3522).


In [ ]:
import numpy as np

cities = np.array([
    [35.6895, 139.6917],   # ?????
    [40.7128, -74.0060],   # ???-????
    [51.5074, -0.1278],    # ??????
    [48.8566, 2.3522],     # ?????
])

city_names = np.array(["?????", "???-????", "??????", "?????"])

cities


2. Преобразуйте значения широты и долготы из градусов в радианы с помощью функции np.radians().


In [ ]:
cities_rad = np.radians(cities)

cities_rad




3. Рассчитайте поэлементную разницу между координатами Токио и других городов в радианах.

In [ ]:
tokyo_coords = cities_rad[0]
coord_diff = cities_rad[1:] - tokyo_coords

for name, diff in zip(city_names[1:], coord_diff):
    print(f"??????? ????????? ????? - {name}: ?????? = {diff[0]:.4f}, ??????? = {diff[1]:.4f} ??????")

coord_diff


## **Задание 2. Операции с DataFrame Pandas и геопространственными данными**


1. Загрузите набор данных о городах мира по следующему URL с помощью Pandas: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv




```python
import pandas as pd
# Pandas поддерживает загрузку данных по прямым ссылкам из сети Интернет
url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"

df = pd.read_csv(url)
```



In [ ]:
import pandas as pd

url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"
df = pd.read_csv(url)

df


2. Отобразите первые 5 строк и проверьте наличие отсутствующих значений.


In [ ]:
display(df.head())

df.isna().sum()


3. Отфильтруйте набор данных, чтобы включить только города с населением более 1 миллиона человек.


In [ ]:
large_cities = df[df["population"] > 1_000_000].copy()

large_cities.head()


4. Сгруппируйте города по странам и рассчитайте общую численность населения для каждой страны.


In [ ]:
country_population = (
    df.groupby("country", as_index=False)["population"]
    .sum()
    .sort_values("population", ascending=False)
)

country_population.head(10)




5. Отсортируйте города по населению в порядке убывания и отобразите первые 10 городов.

In [ ]:
top_10_cities = df.sort_values("population", ascending=False).head(10)

top_10_cities


## **Задание №3. Создание и обработка GeoDataFrames с помощью GeoPandas**


1. Загрузите набор данных о зданиях Нью-Йорка из файла GeoJSON с помощью GeoPandas: https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson

In [ ]:
import geopandas as gpd

buildings_url = "https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson"
buildings = gpd.read_file(buildings_url)

buildings.head()


2. Создайте график контуров зданий и раскрасьте их в зависимости от высоты здания (используйте столбец `height_MS`).


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
buildings.plot(
    column="height_MS",
    cmap="viridis",
    legend=True,
    edgecolor="black",
    linewidth=0.2,
    ax=ax,
)
ax.set_title("??????? ?????? ???-????? ?? ??????")
ax.set_axis_off()
plt.show()


3. Создайте интерактивную карту контуров зданий и раскрасьте их в зависимости от высоты здания (используйте столбец `height_MS`).


In [ ]:
import leafmap

m = leafmap.Map(center=[40.72, -74.00], zoom=14)
m.add_data(
    buildings,
    column="height_MS",
    cmap="viridis",
    scheme="Quantiles",
    k=10,
    add_legend=True,
    layer_name="?????? ??????",
)
m


4. Рассчитайте среднюю высоту зданий (используйте столбец `height_MS`).


In [ ]:
mean_height = buildings["height_MS"].mean()

print(f"??????? ?????? ??????: {mean_height:.2f}")


5. Выберите здания с высотой, превышающей среднюю высоту.


In [ ]:
tall_buildings = buildings[buildings["height_MS"] > mean_height].copy()

tall_buildings.head()





6. Сохраните GeoDataFrame в новый файл GeoJSON.

In [ ]:
output_file = "nyc_tall_buildings.geojson"
tall_buildings.to_file(output_file, driver="GeoJSON")

print(f"???? ????????: {output_file}")


## **Задание №4. Применение NumPy, Pandas и GeoPandas для обработки и анализа пространственных данных**


1. Используйте Pandas для загрузки набора данных о городах мира по следующему URL: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv


In [ ]:
import pandas as pd

url = "https://github.com/opengeos/datasets/releases/download/world/world_cities.csv"
cities_df = pd.read_csv(url)

cities_df.head()


2. Отфильтруйте набор данных, чтобы включить только города с широтой между -40 и 60 (т.е. города, расположенные в Северном полушарии или вблизи экватора).


In [ ]:
filtered_cities = cities_df[
    (cities_df["latitude"] >= -40) &
    (cities_df["latitude"] <= 60)
].copy()

filtered_cities.head()


3. Создайте GeoDataFrame из отфильтрованного набора данных, преобразовав широту и долготу в геометрии.


In [ ]:
import geopandas as gpd

cities_gdf = gpd.GeoDataFrame(
    filtered_cities,
    geometry=gpd.points_from_xy(filtered_cities["longitude"], filtered_cities["latitude"]),
    crs="EPSG:4326",
)

cities_gdf.head()


4. Перепроецируйте GeoDataFrame в проекцию Меркатора (EPSG:3857).


In [ ]:
cities_mercator = cities_gdf.to_crs(epsg=3857)

cities_mercator.head()


5. Рассчитайте расстояние (в метрах) между каждым городом и Парижем.


In [ ]:
from shapely.geometry import Point

paris = gpd.GeoSeries(
    [Point(2.3522, 48.8566)],
    crs="EPSG:4326",
).to_crs(epsg=3857).iloc[0]

cities_mercator["distance_to_paris_m"] = cities_mercator.geometry.distance(paris)
cities_mercator["distance_to_paris_km"] = cities_mercator["distance_to_paris_m"] / 1000

cities_mercator[["name", "country", "distance_to_paris_km"]].sort_values("distance_to_paris_km").head(10)




6. Отобразите города на карте мира, раскрасив точки в зависимости от их расстояния до Парижа.



> 💡 **Подсказка:** Вы можете использовать интерактивную карту `leafmap.Map()` и метод m.add_data(), передав в него ваш GeoDataFrame (предварительно возвращенный в систему координат EPSG:4326).

Основные параметры данного метода:


* column — столбец с рассчитанным расстоянием до Парижа (предварительно разделите значение расстояния на 1000, чтобы легенда отображалась в километрах).
* cmap — цветовая палитра (например, "plasma", "viridis" или "YlOrRd").
* k — количество цветовых интервалов (рекомендуется 10–15 для плавности).
* scheme — алгоритм классификации данных:
  * "Quantiles" (Квантили) — делит города на группы с равным количеством точек в каждой. Это создаст максимальный цветовой контраст: даже если города расположены кучно, вы увидите разницу между ними.
   * "EqualInterval" (Равные интервалы) — делит весь диапазон расстояний на равные отрезки в километрах (например, 0-1000 км, 1000-2000 км и т.д.). Это нагляднее показывает реальный масштаб удаленности.



* установите параметр add_legend=True, чтобы на карте появилась шкала соответствия цветов и расстояний.

In [ ]:
import leafmap

cities_for_map = cities_mercator.to_crs(epsg=4326)

m = leafmap.Map(center=[20, 10], zoom=2)
m.add_data(
    cities_for_map,
    column="distance_to_paris_km",
    cmap="plasma",
    scheme="Quantiles",
    k=12,
    add_legend=True,
    layer_name="?????????? ?? ??????, ??",
)
m
